In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
from torch import Tensor
from torch.nn import Linear
from torch.nn.functional import softmax, cross_entropy
from torch.optim import SGD


import import_ipynb
from common import assert_eq, assert_ge, check_eq, Patient, T # type: ignore


def logistic_regression_nc_sgd_autograd(X: Tensor, Y: Tensor, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with autograd.

    Args:
        X: Input features of shape (Samples, Features)
        Y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    ((s, f), c) = (X.shape, len(set(Y.squeeze().tolist())))

    assert_eq(X.shape, (s, f))
    assert_eq(Y.shape, (s, 1))

    model = Linear(f, c)
    optimizer = SGD(model.parameters(), lr=lr)

    for _ in range(epochs):
        optimizer.zero_grad()

        Z = model(X)
        assert_eq(Z.shape, (s, c))

        L = cross_entropy(Z, Y.squeeze().long())
        assert_eq(L.shape, ())
        
        L.backward()
        optimizer.step()

    return (L.item(), model)


def _test_logistic_regression_nc_sgd_autograd(epochs=2000, lr=0.1) -> None:
    """
    Test the logistic regression implementation on a synthetic dataset of patients. 
    The dataset consists of three classes: healthy (0), viral (1), and bacterial (2). 
    The model is trained on 70 samples and then evaluated on 30 samples (10 per class). 
    The test checks if the model achieves at least 90% accuracy on the evaluation set.
    """

    training_data = T([Patient([0.5, 0.25, 0.25]).data for _ in range(70)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling   

    Y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_nc_sgd_autograd(X, Y, epochs, lr)

    total = 0
    correct = 0

    for d in ([Patient([1.0, 0.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 1.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 0.0, 1.0]).data for _ in range(10)]):
        X = T(d[:-1])
        Y = T(d[-1])

        X[0] /= 100
        X[5] /= 200
        
        total += 1
        correct += check_eq(softmax(model(X), dim=0).argmax(), Y)

    assert_ge(correct / total, 0.9)


if __name__ == "__main__":
    _test_logistic_regression_nc_sgd_autograd()